# 02 — Pré-processamento

**Dimensão 4 da rúbrica — 15 pontos.**

Cada decisão precisa de justificativa escrita. Decidir *não* criar features é
aceitável, desde que o motivo esteja explícito.

In [129]:
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

RAW = Path("..") / "data" / "raw" / "df.csv"
PROCESSED = Path("..") / "data" / "processed"
TARGET = "STATUS"

pd.set_option("display.max_columns", None)

In [130]:
df = pd.read_csv(RAW)
df.head()

,ID,MONTHS_BALANCE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,STATUS
0,5008804,0,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,NaN,2.0,0
1,5008804,-1,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,NaN,2.0,0
2,5008804,-2,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,NaN,2.0,0
3,5008804,-3,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,NaN,2.0,0
4,5008804,-4,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,NaN,2.0,0


## 1. Dados faltantes e Pré-tratamento de colunas

---

*Excluindo as colunas ID e MONTHS_BALANCE*

As colunas ID, MONTHS_BALANCE possuem valores muito diversos e podem atrapalhar em alguns modelos.

In [131]:
df.drop(columns=['ID', 'MONTHS_BALANCE'], inplace=True)
df.head()

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,STATUS
0,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,NaN,2.0,0
1,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,NaN,2.0,0
2,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,NaN,2.0,0
3,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,NaN,2.0,0
4,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,0,0,NaN,2.0,0


---

*Verificando valores duplicados*

In [132]:
df.duplicated().sum()

np.int64(767718)

---

*Excluindo valores duplicados*

In [133]:
df.drop_duplicates(inplace=True)

df.reset_index(drop=True, inplace=True)

df.duplicated().sum()

np.int64(0)

---

*Identificando colunas com valores nulos*

In [134]:
df.isnull().sum()

CODE_GENDER               0
FLAG_OWN_CAR              0
FLAG_OWN_REALTY           0
CNT_CHILDREN              0
AMT_INCOME_TOTAL          0
NAME_INCOME_TYPE          0
NAME_EDUCATION_TYPE       0
NAME_FAMILY_STATUS        0
NAME_HOUSING_TYPE         0
DAYS_BIRTH                0
DAYS_EMPLOYED             0
FLAG_WORK_PHONE           0
FLAG_PHONE                0
FLAG_EMAIL                0
OCCUPATION_TYPE        3081
CNT_FAM_MEMBERS           0
STATUS                    0
dtype: int64

**Interpretação**

Há 3081 nulos na coluna OCCUPATION_TYPE

---

*Tratando os nulos da coluna OCCUPATION_TYPE*

**Decisão:** 

O dicionário de dados informa que, se o número na coluna DAYS_EMPLOYED for positivo, significa que a pessoa está desempregada.

Para essa condição, os nulos da coluna OCCUPATION_TYPE serão substituídos por "Unemployed".

O restante dos nulos serão substituir por "Unknown", pois não se tem informações sobre a ocupação dessas pessoas.

In [135]:
for item in df[df.DAYS_EMPLOYED > 0].index:
  df.loc[item, "OCCUPATION_TYPE"] = "Unemployed"

df.fillna('Unknown', inplace=True)

df.isnull().sum()

CODE_GENDER            0
FLAG_OWN_CAR           0
FLAG_OWN_REALTY        0
CNT_CHILDREN           0
AMT_INCOME_TOTAL       0
NAME_INCOME_TYPE       0
NAME_EDUCATION_TYPE    0
NAME_FAMILY_STATUS     0
NAME_HOUSING_TYPE      0
DAYS_BIRTH             0
DAYS_EMPLOYED          0
FLAG_WORK_PHONE        0
FLAG_PHONE             0
FLAG_EMAIL             0
OCCUPATION_TYPE        0
CNT_FAM_MEMBERS        0
STATUS                 0
dtype: int64

Não há mais valores nulos

---

*Tratando a coluna DAYS_BIRTH*

Esta coluna mostra a idade em dias a partir da coleta dos dados.

In [136]:
# Passando o valor de dias para anos, e transformando em inteiro
df.DAYS_BIRTH = (df.DAYS_BIRTH / -365).astype('int64')

# Renomeando a coluna DAYS_BIRTH para AGE
df.rename(columns={'DAYS_BIRTH': 'AGE'}, inplace=True)

df.head()

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,AGE,DAYS_EMPLOYED,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,STATUS
0,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,32,-4542,1,0,0,Unknown,2.0,0
1,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,58,-1134,0,0,0,Security staff,2.0,0
2,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,52,-3051,0,1,1,Sales staff,1.0,0
3,F,N,Y,0,283500.0,Pensioner,Higher education,Separated,House / apartment,61,365243,0,0,0,Unemployed,1.0,0
4,M,Y,Y,0,270000.0,Working,Higher education,Married,House / apartment,46,-769,1,1,1,Accountants,2.0,0


---

*Tratando a coluna DAYS_EMPLOYED*

Pelo histograma da etapa 1.2, verifica-se que o range dos valores é bem grande a partir do zero.

In [137]:
# Verificando os valores únicos positivos

df.DAYS_EMPLOYED[df.DAYS_EMPLOYED > 0].unique()

array([365243])

**Interpretação**

Nota-se que há apenas 1 valor positivo (365243) para vários clientes (indicando que o cliente está desempregado), o que se trata de uma flag.

In [138]:
# Alterando os valores positivos da coluna DAYS_EMPLOYED para 0, ou seja, não está empregado

df.DAYS_EMPLOYED = df.DAYS_EMPLOYED.apply(lambda x: 0 if x > 0 else x)

# Passando o valor de dias para anos, e transformando em inteiro
df.DAYS_EMPLOYED = (df.DAYS_EMPLOYED / -365).astype('int64')

# Renomeando a coluna DAYS_EMPLOYED para YEARS_EMPLOYED
df.rename(columns={'DAYS_EMPLOYED': 'YEARS_EMPLOYED'}, inplace=True)

df.head()

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,AGE,YEARS_EMPLOYED,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,STATUS
0,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,32,12,1,0,0,Unknown,2.0,0
1,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,58,3,0,0,0,Security staff,2.0,0
2,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,52,8,0,1,1,Sales staff,1.0,0
3,F,N,Y,0,283500.0,Pensioner,Higher education,Separated,House / apartment,61,0,0,0,0,Unemployed,1.0,0
4,M,Y,Y,0,270000.0,Working,Higher education,Married,House / apartment,46,2,1,1,1,Accountants,2.0,0


---

*Criando a coluna de renda per capita e já excluindo as colunas AMT_INCOME_TOTAL e CNT_FAM_MEMBERS*

In [139]:
# Criando a coluna INCOME_PER_MEMBER, que é a renda total dividida pelo número de membros da família
df['INCOME_PER_MEMBER'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']

# Aplicando a transformação logarítmica na coluna INCOME_PER_MEMBER para reduzir a assimetria da distribuição
df['INCOME_PER_MEMBER'] = np.log1p(df['INCOME_PER_MEMBER'])

# Excluindo as colunas AMT_INCOME_TOTAL e CNT_FAM_MEMBERS, pois já temos a coluna INCOME_PER_MEMBER
df.drop(['AMT_INCOME_TOTAL', 'CNT_FAM_MEMBERS'], axis=1, inplace=True)

df.shape

In [140]:
df.head()

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,AGE,YEARS_EMPLOYED,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,STATUS,INCOME_PER_MEMBER
0,M,Y,Y,0,Working,Higher education,Civil marriage,Rented apartment,32,12,1,0,0,Unknown,0,12.272567
1,M,Y,Y,0,Working,Secondary / secondary special,Married,House / apartment,58,3,0,0,0,Security staff,0,10.937579
2,F,N,Y,0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,52,8,0,1,1,Sales staff,0,12.506181
3,F,N,Y,0,Pensioner,Higher education,Separated,House / apartment,61,0,0,0,0,Unemployed,0,12.554971
4,M,Y,Y,0,Working,Higher education,Married,House / apartment,46,2,1,1,1,Accountants,0,11.813037


---

*Tratando a coluna NAME_FAMILY_STATUS*

In [141]:
df.NAME_FAMILY_STATUS = df.NAME_FAMILY_STATUS.replace({
    'Civil marriage': 'Married',
    'Separated': 'Not married', 
    'Single / not married': 'Not married',
    'Widow': 'Not married'
})

df.NAME_FAMILY_STATUS.value_counts()

NAME_FAMILY_STATUS
Married        7587
Not married    2410
Name: count, dtype: int64

---

*Tratando a coluna OCCUPATION_TYPE*

In [142]:
df.OCCUPATION_TYPE = df.OCCUPATION_TYPE.replace(
  {
    'Core staff': 'High Quality staff',
    'Accountants': 'High Quality staff',
    'Managers': 'High Quality staff',
    'High skill tech staff': 'High Quality staff',
    'IT staff': 'High Quality staff',
    'Sales staff': 'Sales and Support',
    'Secrretaries': 'Sales and Support',
    'HR staff': 'Sales and Support',
    'Realty agents': 'Sales and Support',
    'Laborers': 'Blue-collar',
    'Low-skill Laborers': 'Laborers',
    'Drivers': 'Laborers',
    'Medicine staff': 'Health and Care',
    'Cooking staff': 'Health and Care',
    'Waiters/barmen staff': 'Health and Care',
    'Private service staff': 'Health and Care',
    'Security staff': 'Maintenance and Security',
    'Cleaning staff': 'Maintenance and Security'
  }
)

---

*Verificando dados duplicados*

In [143]:
df.duplicated().sum()

np.int64(220)

---

*Excluindo dados duplicados*

In [144]:
df.drop_duplicates(inplace=True)

df.reset_index(drop=True, inplace=True)

df.duplicated().sum()

np.int64(0)

## 2. Definição da variável alvo

A coluna ALVO será a coluna STATUS

In [145]:
y = df[TARGET]

## 3. Normalização / padronização

**Escolha do Escalonador**


Para modelos baseados em árvores não há necessidade de escalonador, porém, para modelos baseados em distância é recomendado sua utilização.

Foi escolhido o **StandardScaler** para as colunas **numéricas**, pois essas colunas apresentam valores bem diferentes.

Aplicando o escalonador elas ficam todas com a mesma escala (com média para 0 e desvio padrão para 1).

---

*Aplicando o **StandardScaler** nas variáveis numéricas de interesse*

In [146]:
df.columns

Index(['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN',
       'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
       'NAME_HOUSING_TYPE', 'AGE', 'YEARS_EMPLOYED', 'FLAG_WORK_PHONE',
       'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'STATUS',
       'INCOME_PER_MEMBER'],
      dtype='str')

In [147]:
# Separando as colunas numéricas de interesse para o escalonamento
numericas = ['CNT_CHILDREN', 'AGE', 'YEARS_EMPLOYED', 'INCOME_PER_MEMBER']

# Instanciando o escalonador
scaler = StandardScaler()

# Escalonando
df_scaled = scaler.fit_transform(df[numericas])

# Transformando em dataframe
df_scaled = pd.DataFrame(df_scaled, columns=numericas)

# Excluindo as colunas numéricas do df original
df.drop(columns=df[numericas], inplace = True)

# Juntando os dataframes
df = pd.concat([df, df_scaled], axis = 1)
df.shape

df.head()

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,STATUS,CNT_CHILDREN,AGE,YEARS_EMPLOYED,INCOME_PER_MEMBER
0,M,Y,Y,Working,Higher education,Married,Rented apartment,1,0,0,Unknown,0,-0.559928,-0.952042,1.053965,1.531624
1,M,Y,Y,Working,Secondary / secondary special,Married,House / apartment,0,0,0,Maintenance and Security,0,-0.559928,1.313257,-0.376569,-0.564940
2,F,N,Y,Commercial associate,Secondary / secondary special,Not married,House / apartment,0,1,1,Sales and Support,0,-0.559928,0.790496,0.418172,1.898509
3,F,N,Y,Pensioner,Higher education,Not married,House / apartment,0,0,0,Unemployed,0,-0.559928,1.574638,-0.853414,1.975132
4,M,Y,Y,Working,Higher education,Married,House / apartment,1,1,1,High Quality staff,0,-0.559928,0.267734,-0.535518,0.809945


## 4. Feature engineering

Há colunas que podem ser binarizadas e há colunas categóricas podem ser tratadas para ficarem com o mesmo padrão das outras colunas.

_____

*Binarizando as colunas FLAG_OWN_CAR, FLAG_OWN_REALTY*

Conforme visto na anáise exploratória de dados, as colunas FLAG_OWN_CAR, FLAG_OWN_REALTY possuem valores dicotômicos (N e Y) que podem ser binarizados (0 e 1)

Essa binarização pode ser feita pelo Label Encoder, porém, neste caso será feito de forma manual.

In [148]:
df.FLAG_OWN_CAR = df.FLAG_OWN_CAR.apply(lambda x: 0 if x=='N' else 1)
df.FLAG_OWN_REALTY = df.FLAG_OWN_REALTY.apply(lambda x: 0 if x=='N' else 1)

----

*Aplicando o **One-Hot Encoder** nas colunas categóricas*

In [149]:
# Separando as colunas categóricas
categoricas = []

for i in df.columns:
  if df[i].dtype == 'str':
    categoricas.append(i)

print(f'Colunas categóricas: {categoricas}')

# Aplicando o hot encoder
hot = []

for i in df.columns:
  hot = pd.get_dummies(df[categoricas], prefix = '')


# Mesclando os  dataframes
df = pd.concat([df, hot], axis=1)

# Excluindo as colunas categóricas originais
df.drop(columns=categoricas, inplace=True)

df.head()

Colunas categóricas: ['CODE_GENDER', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE']


,FLAG_OWN_CAR,FLAG_OWN_REALTY,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,STATUS,CNT_CHILDREN,AGE,YEARS_EMPLOYED,INCOME_PER_MEMBER,_F,_M,_Commercial associate,_Pensioner,_State servant,_Student,_Working,_Academic degree,_Higher education,_Incomplete higher,_Lower secondary,_Secondary / secondary special,_Married,_Not married,_Co-op apartment,_House / apartment,_Municipal apartment,_Office apartment,_Rented apartment,_With parents,_Blue-collar,_Health and Care,_High Quality staff,_Laborers,_Maintenance and Security,_Sales and Support,_Secretaries,_Unemployed,_Unknown
0,1,1,1,0,0,0,-0.559928,-0.952042,1.053965,1.531624,False,True,False,False,False,False,True,False,True,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True
1,1,1,0,0,0,0,-0.559928,1.313257,-0.376569,-0.564940,False,True,False,False,False,False,True,False,False,False,False,True,True,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False
2,0,1,0,1,1,0,-0.559928,0.790496,0.418172,1.898509,True,False,True,False,False,False,False,False,False,False,False,True,False,True,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False
3,0,1,0,0,0,0,-0.559928,1.574638,-0.853414,1.975132,True,False,False,True,False,False,False,False,True,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False
4,1,1,1,1,1,0,-0.559928,0.267734,-0.535518,0.809945,False,True,False,False,False,False,True,False,True,False,False,False,True,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False


In [150]:
df.shape

(9777, 39)

## 5. Salvar dataset tratado

In [151]:
dataset_tratado = df.copy

PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED / "dataset_tratado.csv", index=False)